# Exercise 1: Fix a wrong value with Codex

**Goal:** After this exercise, you can ask Codex to fix a wrong value on a page, and check the fix in the running app.

**Scenario:** A reviewer reports that a case page of the case workbench shows the wrong balance due. You ask Codex to find the cause and fix it. Then you run the app and look at the page.

**Coding agent:** yes. You use Codex in a terminal.

**Where you work:** your own copy of the app, the folder `week-02/day-01/exercise-01-fix-a-wrong-value/reference-app` in the course folder. You run the cells of this notebook and some commands in a terminal. You do not commit: the reset puts the folder back.

**Done when:** cell 3 prints `Done:`, and the page of case CS-2026-0001 shows Balance due $12,250.

**Optional part:** none.

**What you need to know first:**

- A permission prompt is Codex asking you before it does an action, such as a change to a file.
- The balance due of a case is the amount still owed: the assessed amount minus the amount paid.

**What you have:** this exercise's `reference-app` folder is your own copy of the case workbench, with its fictional cases. Codex changes only the copy.

**Start from:** step 1 and cell 1. To start again, open a terminal in the course folder. Run `source .venv/bin/activate`, then `python reset.py exercise-01-fix-a-wrong-value`. The reset puts this folder back as the course supplied it.

### Step 1. See the wrong value

**Before running cell 1:** open **Terminal > New Terminal** in the course folder (the folder containing `pyproject.toml`). Run:

```sh
uv sync
```

This installs the packages for both weeks, including Flask, into the course folder's `.venv`. The first run needs internet access. All three week 2 exercises use this same environment.

**Select the kernel:** click **Select Kernel** at the top right, then **Python Environments**, and select the course folder's `.venv/bin/python`.

**Cannot see it?** Refresh the environment picker after `uv sync`. If it is still missing, run **Python: Select Interpreter** from the Command Palette, choose **Enter interpreter path**, and browse to `.venv/bin/python` in the course folder. Then reopen the notebook's kernel picker and select it. If the notebook already ran with another kernel, restart it and run from cell 1.

Run cell 1. It shows the values that the page of case CS-2026-0001 shows now. You do not need to read its code.

You should see `Balance due: $18,400`. This is the same as `Assessed`, so the page does not subtract the payment.

In [1]:
# 1. See the wrong value. You do not need to read this code.
%cd -q {_dh[0]}/reference-app
import re
import sys

for name in ("app", "cases"):
    sys.modules.pop(name, None)  # read the files as they are now
try:
    import app as workbench
except ImportError:
    print("Flask is missing. Run uv sync in the course folder, select its .venv kernel, and restart from cell 1.")
else:
    page = workbench.app.test_client().get("/cases/CS-2026-0001").get_data(as_text=True)
    for label, value in re.findall(r"<dt>(.*?)</dt><dd>(.*?)</dd>", page):
        if label in ("Assessed", "Paid", "Balance due"):
            print(f"{label}: {value}")

Assessed: $18,400
Paid: $6,150
Balance due: $18,400


### Step 2. Ask Codex to fix the page

Open a terminal: **Terminal > New Terminal**. It opens in the course folder. Run these three commands, one at a time:

```sh
source .venv/bin/activate
cd week-02/day-01/exercise-01-fix-a-wrong-value/reference-app
codex --sandbox read-only --ask-for-approval on-request
```

1. The first command starts the course's shared Python environment in this terminal. Codex then uses the same Python as the app.
2. The second command goes into the folder you work in.
3. The third command starts Codex. Codex can read the files. It must ask you before it changes a file.

If Codex asks if you trust this folder, answer yes. The course gave you this folder.

Write your request to Codex in your own words, and press Enter. It must say these things:

| What your request says | Why |
|---|---|
| What the page of case CS-2026-0001 shows as Balance due, and what it must show | Then Codex can find the cause. The right value is 18,400 minus 6,150. `cases.py`, function `balance_due`: "Return the amount still owed: the assessed amount minus the amount paid." |
| Find the cause and fix it, and do not change the data | The data is right, so the fix must not change it. |

<details><summary>Stuck? One way to say it</summary>

`The page of case CS-2026-0001 shows Balance due $18,400. It must show $12,250, which is Assessed minus Paid. Find the cause and fix it. Do not change the data.`

</details>

Codex can ask you before an action. Read the action. Then answer:

| The prompt asks to | Your answer | Why |
|---|---|---|
| Change a file in `reference-app`, such as a page in `templates/` | Yes | The fix is a change to your copy of the app. |
| Run `pytest` | Yes | It runs the app's tests and changes nothing outside the folder. |
| Start the app with `flask` | No | The app keeps running and Codex waits. You start the app yourself in step 4. |
| Use the network, or change a file outside `reference-app` | No | This fix needs neither. "Before you answer a prompt, read the command. Ask why this task needs it." |
| Something else | Yes only if it reads or changes files in your copy | The task needs nothing outside your copy. |

When Codex reports that it fixed the page, type `/quit` and press Enter. This stops Codex.

### Step 3. See what Codex changed

Run cell 2. It shows each file that Codex changed, then each changed line. A line that starts with `-` is the old line. A line that starts with `+` is the new line.

You should see `templates/case.html`: its Balance due row now shows `case.balance_due`, not `case.assessed`. The calculation in `cases.py` was right. Only the page showed the wrong value.

In [2]:
# 2. See what Codex changed. You do not need to read this code.
%cd -q {_dh[0]}/reference-app
!git diff --quiet -- . && echo "No change yet."
!git diff --name-only -- .
!git diff -U0 --no-color -- . | grep '^[-+][^-+]' | head -n 10

week-02/day-01/exercise-01-fix-a-wrong-value/reference-app/templates/case.html
-  <dt>Balance due</dt><dd>{{ case.assessed | money }}</dd>
+  <dt>Balance due</dt><dd>{{ case.balance_due | money }}</dd>


### Step 4. Run the app and look at the page

In the terminal of step 2, which is still in your copy with the environment on, run:

```sh
flask --app app run
```

This command starts the app. It prints `Running on http://127.0.0.1:5000`. Open that link (Ctrl+click), then click `CS-2026-0001` in the case list. If it says that the address is already in use, stop the app in your other terminal with Ctrl+C. If the link does not open, see the course README, "If week 2 setup needs attention".

You should see `Balance due $12,250`. Stop the app with Ctrl+C in the terminal.

### Step 5. Check

Run cell 3. It checks every case page, the data and the app's tests. It prints `Done:` when three things are true:

- Each case page shows Balance due as Assessed minus Paid.
- The data has not changed.
- The app's tests pass.

Otherwise it prints `Not yet:` and each thing that is still wrong. You do not need to read its code.

In [3]:
# 3. Check. You do not need to read this code.
%cd -q {_dh[0]}/reference-app
import json
import re
import subprocess
import sys


def check() -> list[str]:
    """Return what is left: one line for each thing that is not yet right."""
    left = []
    if subprocess.run(["git", "diff", "--quiet", "HEAD", "--", "data"]).returncode:
        left.append("data/cases.json changed. Run the reset (Start from) and ask Codex again.")
    for name in ("app", "cases"):
        sys.modules.pop(name, None)  # read the files as they are now
    try:
        import app as workbench
    except ImportError:
        return ["Flask is missing. Run uv sync in the course folder, select its .venv kernel, and restart from cell 1."]
    client = workbench.app.test_client()
    wrong = []
    for case in json.load(open("data/cases.json", encoding="utf-8")):
        page = client.get(f"/cases/{case['case_id']}").get_data(as_text=True)
        shown = dict(re.findall(r"<dt>(.*?)</dt><dd>(.*?)</dd>", page)).get("Balance due")
        if shown != f"${case['assessed'] - case['paid']:,}":
            wrong.append(f"{case['case_id']} shows {shown}")
    if wrong:
        left.append(f"Pages with the wrong Balance due: {len(wrong)} ({wrong[0]}). "
                    "Ask Codex again, as step 2 says.")
    tests = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "--color=no"],
                           capture_output=True, text=True)
    if tests.returncode:
        last = (tests.stdout.strip() or tests.stderr.strip()).splitlines()[-1]
        left.append(f"The app's tests do not pass ({last}). Ask Codex to fix what it changed.")
    return left


left = check()
print(f"Not yet: {len(left)} left." if left else "Done: every case page shows Balance due as "
      "Assessed minus Paid (CS-2026-0001: $12,250), the data is as supplied, and the tests pass.")
for line in left:
    print("- " + line)

Done: every case page shows Balance due as Assessed minus Paid (CS-2026-0001: $12,250), the data is as supplied, and the tests pass.


### Afterwards, together

1. The app's tests passed before the fix too. Why did they not find the wrong value?
2. In the Auto setting, Codex "can read files, make edits, and run commands in the working directory automatically." When would you start Codex in Auto, not read-only, and why?
3. You did not let Codex start the app. On a project with real taxpayer data, which other prompts would you refuse, and why?

<details><summary>Answer</summary>

The change: in `templates/case.html`, the Balance due row shows `case.balance_due`, not `case.assessed`. The page showed the assessed amount in that row. The app already calculates the balance due in `cases.py`, and the page gets it as `case.balance_due`. So the fix uses that value and changes no calculation.

The prompts: yes to a change in your copy, because the fix is a change to the app. No to starting the app: `flask --app app run` keeps running, so Codex waits for it. No to the network or a file outside `reference-app`: the fix needs neither.

The fixed app is in [`answers/exercise-01-fix-a-wrong-value/reference-app`](../answers/exercise-01-fix-a-wrong-value/reference-app/). Each change and its reason is also in [`answers/exercise-01-fix-a-wrong-value/decisions.md`](../answers/exercise-01-fix-a-wrong-value/decisions.md).

</details>